# Fast R-CNN Training Notebook for Kaggle

This notebook is designed to train the **Fast R-CNN model** on a synthetic MNIST multi-digit dataset. By running this notebook on **Kaggle** with a GPU accelerator, training will complete significantly faster (under 3 minutes) compared to training locally on a CPU.

## Kaggle Setup Instructions
1. **Create a Kaggle account** (if you don't have one) and log in.
2. Click on **"New Notebook"** in Kaggle.
3. Go to the File menu inside the notebook editor, select **"Upload notebook"**, and choose this file (`train_fast_rcnn_kaggle.ipynb`).
4. In the right-hand panel of the Kaggle editor, expand **"Accelerator"** and select **"GPU T4 x2"** or **"GPU P100"** (ensure GPU accelerator is enabled).
5. Ensure **"Internet on"** is selected in the Settings section of the right-hand panel (needed to download the standard MNIST dataset).
6. Run all cells in this notebook. The final model weights will be saved as `best_model.pth` in the `/kaggle/working/` output directory, which can be downloaded directly from the Kaggle file navigator on the right.

In [ ]:
# Step 1: Imports and Device Setup
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

# Select GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Step 2: Model Architecture Definitions (ROIPool & FastRCNN)
# Must match the local implementation exactly so that saved weight keys align.

class ROIPool(nn.Module):
    """
    Region of Interest (RoI) Pooling Layer.
    Extracts fixed-size feature maps (e.g., 7x7) from varying-sized region proposals.
    """
    def __init__(self, output_size, spatial_scale):
        super(ROIPool, self).__init__()
        self.output_size = output_size  # (pooled_h, pooled_w)
        self.spatial_scale = spatial_scale

    def forward(self, features, rois):
        N = rois.size(0)
        C, H, W = features.size(1), features.size(2), features.size(3)
        out_h, out_w = self.output_size
        
        if N == 0:
            return torch.zeros(0, C, out_h, out_w, device=features.device, dtype=features.dtype)
            
        output = torch.zeros(N, C, out_h, out_w, device=features.device, dtype=features.dtype)
        
        for i in range(N):
            roi = rois[i]
            batch_idx = int(roi[0].item())
            
            x1 = roi[1] * self.spatial_scale
            y1 = roi[2] * self.spatial_scale
            x2 = roi[3] * self.spatial_scale
            y2 = roi[4] * self.spatial_scale
            
            x1_idx = int(torch.round(x1).item())
            y1_idx = int(torch.round(y1).item())
            x2_idx = int(torch.round(x2).item())
            y2_idx = int(torch.round(y2).item())
            
            x1_idx = max(0, min(x1_idx, W - 1))
            y1_idx = max(0, min(y1_idx, H - 1))
            x2_idx = max(0, min(x2_idx, W - 1))
            y2_idx = max(0, min(y2_idx, H - 1))
            
            roi_w = max(1, x2_idx - x1_idx + 1)
            roi_h = max(1, y2_idx - y1_idx + 1)
            
            crop = features[batch_idx, :, y1_idx:y1_idx+roi_h, x1_idx:x1_idx+roi_w]
            
            crop_cpu = crop.cpu().unsqueeze(0)
            pooled_cpu = F.adaptive_max_pool2d(crop_cpu, self.output_size)
            output[i] = pooled_cpu.squeeze(0).to(features.device)
            
        return output

class FastRCNN(nn.Module):
    """
    Fast R-CNN Model Architecture for MNIST digit detection.
    Downsamples the 256x256 image by a factor of 8 to a 32x32 feature map.
    """
    def __init__(self, num_classes=11, pool_size=(7, 7), spatial_scale=0.125):
        super(FastRCNN, self).__init__()
        
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 128x128
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 64x64
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32x32
        )
        
        self.roi_pool = ROIPool(output_size=pool_size, spatial_scale=spatial_scale)
        
        self.classifier = nn.Sequential(
            nn.Linear(64 * pool_size[0] * pool_size[1], 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(256, num_classes)
        )
        
        self.bbox_regressor = nn.Sequential(
            nn.Linear(64 * pool_size[0] * pool_size[1], 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 4)
        )
        
    def forward(self, x, rois):
        feat_map = self.backbone(x)
        pooled_feats = self.roi_pool(feat_map, rois)
        pooled_feats_flat = pooled_feats.view(pooled_feats.size(0), -1)
        
        cls_logits = self.classifier(pooled_feats_flat)
        bbox_offsets = self.bbox_regressor(pooled_feats_flat)
        
        return feat_map, cls_logits, bbox_offsets

In [ ]:
# Step 3: Dataset Generation (SyntheticMNISTDataset)

class SyntheticMNISTDataset(torch.utils.data.Dataset):
    """
    Generates synthetic 256x256 images containing 1-3 random MNIST digits.
    This serves as our multi-object dataset for Fast R-CNN object detection.
    """
    def __init__(self, mnist_dataset, num_samples=500, img_size=256):
        self.mnist = mnist_dataset
        self.num_samples = num_samples
        self.img_size = img_size
        
    def __len__(self):
        return self.num_samples
        
    def __getitem__(self, idx):
        img = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        num_digits = np.random.randint(1, 4)
        
        gt_boxes = []
        gt_classes = []
        
        attempts = 0
        placed_digits = 0
        while placed_digits < num_digits and attempts < 100:
            attempts += 1
            mnist_idx = np.random.randint(0, len(self.mnist))
            digit_img, digit_label = self.mnist[mnist_idx]
            
            if torch.is_tensor(digit_img):
                digit_img = digit_img.detach().cpu().numpy()
            if isinstance(digit_img, np.ndarray):
                if digit_img.ndim == 3 and digit_img.shape[0] == 1:
                    digit_img = digit_img.squeeze(0)
            else:
                digit_img = np.array(digit_img, dtype=np.float32)
            
            x = np.random.randint(15, self.img_size - 43)
            y = np.random.randint(15, self.img_size - 43)
            box = [x, y, x + 28, y + 28]
            
            overlap = False
            for placed_box in gt_boxes:
                if self._compute_iou(box, placed_box) > 0.0:
                    overlap = True
                    break
                    
            if not overlap:
                img[y:y+28, x:x+28] = np.maximum(img[y:y+28, x:x+28], digit_img)
                gt_boxes.append(box)
                gt_classes.append(digit_label)
                placed_digits += 1
                
        if placed_digits == 0:
            mnist_idx = np.random.randint(0, len(self.mnist))
            digit_img, digit_label = self.mnist[mnist_idx]
            if torch.is_tensor(digit_img):
                digit_img = digit_img.detach().cpu().numpy()
            if isinstance(digit_img, np.ndarray):
                if digit_img.ndim == 3 and digit_img.shape[0] == 1:
                    digit_img = digit_img.squeeze(0)
            else:
                digit_img = np.array(digit_img, dtype=np.float32)
            x, y = 114, 114
            box = [x, y, x + 28, y + 28]
            img[y:y+28, x:x+28] = digit_img
            gt_boxes.append(box)
            gt_classes.append(digit_label)
            
        img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0)  # (1, 256, 256)
        img_tensor = (img_tensor - 0.1307) / 0.3081
        
        return img_tensor, torch.tensor(gt_boxes, dtype=torch.float32), torch.tensor(gt_classes, dtype=torch.long)
        
    def _compute_iou(self, boxA, boxB):
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[2], boxB[2])
        yB = min(boxA[3], boxB[3])
        
        interArea = max(0, xB - xA) * max(0, yB - yA)
        boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
        boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
        unionArea = boxAArea + boxBArea - interArea
        
        if unionArea == 0:
            return 0.0
        return interArea / unionArea

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    gt_boxes = [item[1] for item in batch]
    gt_classes = [item[2] for item in batch]
    return images, gt_boxes, gt_classes

In [ ]:
# Step 4: ROI Sampling & Loss Definitions

def compute_single_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    
    interArea = max(0.0, xB - xA) * max(0.0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    unionArea = boxAArea + boxBArea - interArea
    
    if unionArea == 0.0:
        return 0.0
    return (interArea / unionArea).item()

def generate_training_rois(gt_boxes, gt_classes, img_size=256, max_pos=16, max_neg=48):
    device = gt_boxes.device
    num_gt = gt_boxes.size(0)
    
    rois = []
    labels = []
    bbox_targets = []
    
    pos_count = 0
    if num_gt > 0:
        for i in range(num_gt):
            rois.append(gt_boxes[i])
            labels.append(gt_classes[i])
            bbox_targets.append(torch.tensor([0.0, 0.0, 0.0, 0.0], device=device))
            pos_count += 1
            
        attempts = 0
        while pos_count < max_pos and attempts < 150:
            attempts += 1
            idx = torch.randint(0, num_gt, (1,)).item()
            gt_box = gt_boxes[idx]
            gt_cls = gt_classes[idx]
            
            x1, y1, x2, y2 = gt_box[0].item(), gt_box[1].item(), gt_box[2].item(), gt_box[3].item()
            w = x2 - x1
            h = y2 - y1
            
            dx_shift = np.random.uniform(-4, 4)
            dy_shift = np.random.uniform(-4, 4)
            dw_scale = np.random.uniform(-0.15, 0.15)
            dh_scale = np.random.uniform(-0.15, 0.15)
            
            px1 = x1 + dx_shift
            py1 = y1 + dy_shift
            pw = w * (1.0 + dw_scale)
            ph = h * (1.0 + dh_scale)
            px2 = px1 + pw
            py2 = py1 + ph
            
            px1 = max(0.0, min(px1, img_size - 2.0))
            py1 = max(0.0, min(py1, img_size - 2.0))
            px2 = max(px1 + 2.0, min(px2, img_size - 1.0))
            py2 = max(py1 + 2.0, min(py2, img_size - 1.0))
            
            prop_box = torch.tensor([px1, py1, px2, py2], device=device)
            iou = compute_single_iou(prop_box, gt_box)
            
            if iou >= 0.5:
                rois.append(prop_box)
                labels.append(gt_cls)
                
                pw_val = px2 - px1
                ph_val = py2 - py1
                px_ctr = px1 + pw_val / 2.0
                py_ctr = py1 + ph_val / 2.0
                
                gt_w = x2 - x1
                gt_h = y2 - y1
                gt_x_ctr = x1 + gt_w / 2.0
                gt_y_ctr = y1 + gt_h / 2.0
                
                tx = (gt_x_ctr - px_ctr) / pw_val
                ty = (gt_y_ctr - py_ctr) / ph_val
                tw = np.log(gt_w / pw_val)
                th = np.log(gt_h / ph_val)
                
                bbox_targets.append(torch.tensor([tx, ty, tw, th], device=device, dtype=torch.float32))
                pos_count += 1
                
    neg_count = 0
    attempts = 0
    while neg_count < max_neg and attempts < 300:
        attempts += 1
        pw = np.random.uniform(20, 60)
        ph = np.random.uniform(20, 60)
        px1 = np.random.uniform(0, img_size - pw - 1)
        py1 = np.random.uniform(0, img_size - ph - 1)
        px2 = px1 + pw
        py2 = py1 + ph
        
        prop_box = torch.tensor([px1, py1, px2, py2], device=device)
        
        max_iou = 0.0
        for i in range(num_gt):
            iou = compute_single_iou(prop_box, gt_boxes[i])
            if iou > max_iou:
                max_iou = iou
                
        if max_iou < 0.2:
            rois.append(prop_box)
            labels.append(torch.tensor(10, device=device))  # 10 is Background
            bbox_targets.append(torch.tensor([0.0, 0.0, 0.0, 0.0], device=device))
            neg_count += 1
            
    if len(rois) == 0:
        rois.append(torch.tensor([0.0, 0.0, 28.0, 28.0], device=device))
        labels.append(torch.tensor(10, device=device))
        bbox_targets.append(torch.tensor([0.0, 0.0, 0.0, 0.0], device=device))
        
    rois = torch.stack(rois)
    labels = torch.stack(labels)
    bbox_targets = torch.stack(bbox_targets)
    
    return rois, labels, bbox_targets

def fast_rcnn_loss(cls_logits, bbox_offsets, labels, bbox_targets):
    loss_cls = F.cross_entropy(cls_logits, labels)
    
    foreground_mask = (labels < 10).float()
    loss_bbox_full = F.smooth_l1_loss(bbox_offsets, bbox_targets, reduction='none')
    loss_bbox_masked = loss_bbox_full.sum(dim=1) * foreground_mask
    
    num_foreground = foreground_mask.sum()
    if num_foreground > 0:
        loss_bbox = loss_bbox_masked.sum() / num_foreground
    else:
        loss_bbox = loss_bbox_masked.sum() * 0.0
        
    total_loss = loss_cls + 1.0 * loss_bbox
    return total_loss, loss_cls, loss_bbox

def compute_metrics(preds, targets, num_classes=11):
    preds = np.array(preds)
    targets = np.array(targets)
    accuracy = np.mean(preds == targets) * 100.0
    
    f1_classes = []
    for c in range(num_classes):
        tp = np.sum((preds == c) & (targets == c))
        fp = np.sum((preds == c) & (targets != c))
        fn = np.sum((preds != c) & (targets == c))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_classes.append(f1)
        
    macro_f1 = np.mean(f1_classes)
    return round(accuracy, 2), round(macro_f1, 4)

In [ ]:
# Step 5: Training & Evaluation Functions

def train_fast_rcnn(model, train_loader, test_loader, optimizer, epochs=5):
    print("Starting training loop...")
    total_batches = len(train_loader)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        all_preds = []
        all_targets = []
        
        for batch_idx, (images, gt_boxes_list, gt_classes_list) in enumerate(train_loader):
            images = images.to(device)
            
            batch_rois = []
            batch_labels = []
            batch_bbox_targets = []
            
            for b_idx in range(images.size(0)):
                gt_boxes = gt_boxes_list[b_idx].to(device)
                gt_classes = gt_classes_list[b_idx].to(device)
                
                rois, labels, bbox_targets = generate_training_rois(gt_boxes, gt_classes, img_size=256)
                
                batch_col = torch.full((rois.size(0), 1), b_idx, dtype=torch.float32, device=device)
                rois_with_batch = torch.cat([batch_col, rois], dim=1)
                
                batch_rois.append(rois_with_batch)
                batch_labels.append(labels)
                batch_bbox_targets.append(bbox_targets)
                
            batch_rois = torch.cat(batch_rois, dim=0)
            batch_labels = torch.cat(batch_labels, dim=0)
            batch_bbox_targets = torch.cat(batch_bbox_targets, dim=0)
            
            optimizer.zero_grad()
            _, cls_logits, bbox_offsets = model(images, batch_rois)
            loss, _, _ = fast_rcnn_loss(cls_logits, bbox_offsets, batch_labels, batch_bbox_targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = cls_logits.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(batch_labels.cpu().numpy())
            
            if (batch_idx + 1) % 50 == 0 or batch_idx == total_batches - 1:
                cur_loss = running_loss / (batch_idx + 1)
                cur_acc, cur_f1 = compute_metrics(all_preds, all_targets)
                print(f"Epoch {epoch+1}/{epochs} | Batch {batch_idx+1}/{total_batches} | Loss: {cur_loss:.4f} | Acc: {cur_acc}% | F1: {cur_f1}")
                
        epoch_loss = running_loss / total_batches
        epoch_acc, epoch_f1 = compute_metrics(all_preds, all_targets)
        print(f"--- Epoch {epoch+1} Complete: Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc}%, F1: {epoch_f1} ---\n")
        
    # Testing Phase
    print("Starting testing evaluation...")
    model.eval()
    test_preds = []
    test_targets = []
    
    with torch.no_grad():
        for images, gt_boxes_list, gt_classes_list in test_loader:
            images = images.to(device)
            batch_rois = []
            batch_labels = []
            for b_idx in range(images.size(0)):
                gt_boxes = gt_boxes_list[b_idx].to(device)
                gt_classes = gt_classes_list[b_idx].to(device)
                rois, labels, _ = generate_training_rois(gt_boxes, gt_classes, img_size=256)
                batch_col = torch.full((rois.size(0), 1), b_idx, dtype=torch.float32, device=device)
                batch_rois.append(torch.cat([batch_col, rois], dim=1))
                batch_labels.append(labels)
                
            if len(batch_rois) == 0: continue
            batch_rois = torch.cat(batch_rois, dim=0)
            batch_labels = torch.cat(batch_labels, dim=0)
            
            _, cls_logits, _ = model(images, batch_rois)
            _, predicted = cls_logits.max(1)
            test_preds.extend(predicted.cpu().numpy())
            test_targets.extend(batch_labels.cpu().numpy())
            
    test_acc, test_f1 = compute_metrics(test_preds, test_targets)
    print(f"==============================================")
    print(f"Test Set Evaluation - Accuracy: {test_acc}%, Macro F1: {test_f1}")
    print(f"==============================================")
    
    # Save Model Weights
    save_path = "best_model.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Model weights saved successfully to: {os.path.abspath(save_path)}")

In [ ]:
# Step 6: Load Data and Start Training
# Note: In Kaggle, we can train on a larger dataset (e.g. 5000 train samples, 1000 test samples)
# and run for more epochs (e.g. 10 epochs) to achieve high-performance accuracy!

# Create data directory
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

# Load original MNIST dataset (will be downloaded automatically)
transform = transforms.Compose([transforms.ToTensor()])
mnist_train = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

# Configuration (Feel free to scale up these parameters on Kaggle!)
TRAIN_SAMPLES = 5000  # Local default was 600
TEST_SAMPLES = 1000   # Local default was 150
BATCH_SIZE = 16       # Local default was 8
EPOCHS = 10           # Local default was 3
LEARNING_RATE = 0.001

print(f"Generating Synthetic Datasets... (Train: {TRAIN_SAMPLES}, Test: {TEST_SAMPLES})")
train_dataset = SyntheticMNISTDataset(mnist_train, num_samples=TRAIN_SAMPLES, img_size=256)
test_dataset = SyntheticMNISTDataset(mnist_test, num_samples=TEST_SAMPLES, img_size=256)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# Initialize Model and Optimizer
model = FastRCNN(num_classes=11).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Run Training
train_fast_rcnn(model, train_loader, test_loader, optimizer, epochs=EPOCHS)

## How to Download Weights and Load Them Locally

Once training completes, you will see a `best_model.pth` file in the Output files section of your Kaggle environment:

1. In the right side pane of the Kaggle editor, locate the **"Output"** folder under the **"Data"** tab (or check the files explorer).
2. Locate the file **`best_model.pth`** under `/kaggle/working/`.
3. Click on the three dots next to `best_model.pth` and choose **"Download"**.
4. Move the downloaded `best_model.pth` weight file into the **`checkpoints`** folder of your local project:
   `.../2-Region-Proposals(Fast_R-CNN_Idea)/checkpoints/best_model.pth`
5. Restart your local web app (or simply trigger detection in the UI) to load the new premium weights automatically!